# **Dataset Maker**

Follow this notebook to transform your data into a dataset that can be used to train the model.

## 📋 **1. Required Data**

RNeNcodec requires three types of data: **audio files**, **annotations**, and **parameter specifications**. Each one is explained below.

### **1.1 🎵 Audio Files**

Most audio file formats and sampling rates are supported, although the final model will always operate at 24kHz.

**Supported Audio Formats**: `.wav`, `.mp3`, `.flac`, `.aac`, `.ogg`, `.m4a`, `.wma`, `.aiff`, `.au`, `.ra`, `.3gp`, `.amr`, `.ac3`, `.dts`, `.ape`, `.mka`, `.opus`

### **1.2 📊 Annotations (CSV Files)**

Each audio file **must** have a corresponding CSV file with the **exact same base name**:

| Audio File | CSV File | Status |
|------------|----------|---------|
| `piano.wav` | `piano.csv` | ✅ Correct |
| `flute.mp3` | `flute_annotations.csv` | ⛔ Incorrect - name mismatch! |

The CSV annotation files must follow this specific structure:

- **Header row** with parameter names (must match those in `parameters.json`)
- **One row per frame** (75 frames per second)
- **Continuous parameters**: Numeric values (will be normalized to [0,1] range)
- **Categorical parameters**: Non-negative integers representing each class (0, 1, 2, ...)

Example CSV:

| saturation | reverb | instrument |
|------------|--------|------------|
| 10.5       | 40.3   | 0          |
| 10.0       | 40.32  | 1          |
| 9.8        | 40.25  | 0          |
| ...        | ...    | ...        |

### **1.3 ⚙️ Parameter Specifications**

Information about the parameters to be controlled must be stored in a `parameters.json` file. Each parameter must specify its **name** and **type** (either `continuous` or `class`). Additional features will depepnd on the type of the parameter as follow:

**🔢 Continuous Parameters** (numeric values like tempo, volume, saturation):
- `min`/`max`: Range used for normalization
- `unit`: Physical unit (e.g., bpm, %, dB)

**🏷️ Class Parameters** (categorical values like instrument type, genre):
- `classes`: Ordered list of class names (order determines integer mapping)

#### Example `parameters.json`:

```json
{
    "parameter_1": {"name": "saturation", "type": "continuous", "unit": "dB", "min": 0,   "max": 12}, 
    "parameter_2": {"name": "reverb",     "type": "continuous", "unit": "%",  "min": 0,   "max": 100},
    "parameter_3": {"name": "instrument", "type": "class",      "classes": ["piano", "flute"]}
}
```
## **📁 2. Required Data Structure**

Before running this notebook on your data, make sure it is structured as follows:

**Option 1:** Simple Structure (all data together)

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── piano.wav                # Audio files (various formats supported)
    ├── piano.csv                # Corresponding CSV annotations
    ├── flute.mp3                # More audio files...
    ├── flute.csv                # More CSV files...
    └── ...
```

**Option 2:** Split Structure (separate train/validation/test sets)
If you want to use specific data splits for training, validation, and testing.

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── train/
    │   ├── piano.wav            # Training audio files
    │   ├── piano.csv            # Training CSV annotations
    │   └── ...
    ├── validation/
    │   ├── flute.mp3            # Validation audio files
    │   ├── flute.csv            # Validation CSV files
    │   └── ...
    └── test/
        ├── violin.mp3           # Test audio files
        ├── violin.csv           # Test CSV files
        └── ...
```

## **🔄 3. Dataset Creation Pipeline**

Once your data is properly arranged, follow this notebook to create your dataset. The full pipeline consists of five steps:

- **📊 Visualization** - Analyze your raw data structure and parameters to verify everything is in order
- **🔧 Normalization** - Standardize format and normalize your data
- **🎵 EnCodec Encoding** - Convert audio to a compressed latent format that RNeNcodec can process
- **📄 Sidecar Creation**- Align parameters with audio frames (75 fps)
- **🤗 HuggingFace Dataset** - Package everything into a training-ready format

Let's get started! 👇

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "quickstart") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

dataset_path = "path/to/dataset" # Path to dataset directory

### **3.1 📊 Visualization**

Visualize your data and make sure everything is alright.

In [ ]:
# Import visualization functions
from dataprep.step_0_visualization import quick_analyze

# 🔍 ANALYZE ENTIRE DATASET
quick_analyze(dataset_path, individual_plot_selector=False) # Set individual_plot_selector=True to choose which files to plot.

### **3.2 🔧 Normalization**

This step prepares your audio files by:
1. **Resampling** all audio to 24kHz mono (required by EnCodec)
2. **RMS Normalization** (optional) to ensure consistent loudness across your dataset

In [ ]:
from dataprep.step_1_normalization import quick_normalize

quick_normalize(dataset_path, apply_rms_normalization=True)

### **3.3 🎵 EnCodec Encoding**

In [ ]:
from dataprep.step_2_encodec import quick_encode

quick_encode(dataset_path, bandwidth=6.0, device="cpu")  # Force 8 codebooks and CPU

### **3.4 📄 Sidecar Creation**

In [ ]:
from dataprep.step_3_sidecars import quick_create_sidecars

results = quick_create_sidecars(dataset_path)

### **3.5 🤗 HuggingFace Dataset**

In [ ]:
from dataprep.step_4_HF import quick_create_dataset

results = quick_create_dataset(dataset_path)